In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar
import pandas as pd
import cartopy.util
import xesmf as xe
import src.XRO_utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(r"/glade/work/kcarr/lme_data")

## Funcs

In [ ]:
def prep_seasonal(y):
    """prepare data for EOF analysis"""

    ## resample to seasonal
    y_ = y.resample({"time": "QS-JAN"}).mean()

    ## get year start
    yrs = y_.time.dt.year.values
    mths = y_.time.dt.month.values
    year_start = np.array([y - 1 if (m < 7) else y for (y, m) in zip(yrs, mths)])

    ## get season
    season_dict = {1: "JFM", 4: "AMJ", 7: "JAS", 10: "OND"}
    season = np.array([season_dict[m] for m in mths])

    ## create multi-index
    new_idx = pd.MultiIndex.from_arrays([year_start, season], names=["y0", "season"])

    ## convert to xr
    new_idx_xr = xr.Coordinates.from_pandas_multiindex(new_idx, dim="time")

    ## add to original data
    y_ = y_.assign_coords(new_idx_xr).unstack("time")

    ## re-order seasons
    y_ = y_.reindex({"season": ["JAS", "OND", "JFM", "AMJ"]})

    return y_.isel(y0=slice(1, -1))


def prep_and_trim(x):
    """rename coordinates and trim in time"""

    # ## time coordinates for trimming
    time_idx = dict(time=slice("1949-01", "2020-12"))

    ## rename spatial coords
    coord_dict = dict(lat="latitude", lon="longitude")

    return x.sel(time_idx).rename(coord_dict)

## Load data

#### CESM2

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
sst = xr.open_dataset(
    DATA_FP / "snr" / "sst_split_deseason.nc", decode_times=time_coder
)
pr = xr.open_dataset(DATA_FP / "snr" / "pr_split_deseason.nc", decode_times=time_coder)
pr = pr.assign_coords({"time": sst.time})

## trim
sst = prep_and_trim(sst)
pr = prep_and_trim(pr)

## regrid
GRID = pr.isel(member=0, time=0)


def regrid_like_pr(x):
    """regrid like precip data"""
    regridder = xe.Regridder(x, GRID, "bilinear")
    return regridder(x)


## regrid sst
sst = regrid_like_pr(sst)

In [ ]:
## put in seasonal form
sst = prep_seasonal(sst)
pr = prep_seasonal(pr)

## clims/anomalies
clim = xr.merge(
    [
        sst["forced"].mean("y0").rename("sst"),
        pr["forced"].mean("y0").rename("pr"),
    ]
)

anom = xr.merge(
    [
        sst["anom"].rename("sst"),
        pr["anom"].rename("pr"),
    ]
)

### Reference data

#### ERRST

In [ ]:
## reference data
REF_FP = pathlib.Path("/glade/campaign/cgd/cas/asphilli/CVDP-OBS/")
sst_ref = xr.open_dataset(REF_FP / "ersstv5.185401-202412.nc")["ssta"]
total_ref = xr.open_dataset(REF_FP / "ersstv5.185401-202412.nc")["sst"]
pr_ref = xr.open_dataset(REF_FP / "gpcp.mon.mean.197901-202403.nc")["precip"]
# pr_ref = xr.open_dataarray(DATA_FP / "precip_gpcc_lores.nc")

## convert from mm to mm/day
# days_per_month = 30
# pr_ref = pr_ref / days_per_month

## assign time coord
sst_ref["time"] = xr.date_range(start="1854-01", end="2024-12", freq="MS")
total_ref["time"] = xr.date_range(start="1854-01", end="2024-12", freq="MS")

#### prep
do_prep = lambda x: prep_seasonal(regrid_like_pr(prep_and_trim(x)))
sst_ref = do_prep(sst_ref)
total_ref = do_prep(total_ref)
pr_ref = do_prep(pr_ref)
# pr_ref = prep_seasonal(prep_and_trim(pr_ref))

## remove anomalies
clim_ref = xr.merge(
    [
        total_ref.mean("y0").rename("sst"),
        pr_ref.mean("y0").rename("pr"),
    ]
)

center = lambda x: x - x.mean("y0")
anom_ref = xr.merge(
    [
        center(sst_ref).rename("sst"),
        center(pr_ref).rename("pr"),
    ]
)

## Make composites

In [ ]:
## specify index fn
idx_fn = lambda x: src.utils.get_nino3(x.sel(season="OND"))
# idx_fn = lambda x: src.utils.get_nino34(x.sel(season="JFM"))

## Get index
idx = idx_fn(anom["sst"])
idx_ref = idx_fn(anom_ref["sst"])

In [ ]:
def get_warm_composite(data, idx, q=0.75):
    """get warm composite"""

    ## get masked data
    data_ = data.where(idx > idx.quantile(q=q, dim="y0"))

    return data_.mean("y0").drop_vars("quantile")


def get_cold_composite(data, idx, q=0.75):
    """get warm composite"""

    ## get masked data
    data_ = data.where(idx < idx.quantile(q=(1 - q), dim="y0"))

    return data_.mean("y0").drop_vars("quantile")


def get_composites(data, idx, q=0.75):
    """get composites"""

    kwargs = dict(data=data, idx=idx, q=q)

    ## get rename dicts
    names_warm = {n: f"{n}_warm" for n in list(data)}
    names_cold = {n: f"{n}_cold" for n in list(data)}
    comps = xr.merge(
        [
            get_warm_composite(**kwargs).rename(names_warm),
            get_cold_composite(**kwargs).rename(names_cold),
        ],
    )

    return comps


## get composites
q = 0.8
comps_ref = get_composites(anom_ref, idx=idx_ref, q=q)
comps = get_composites(anom, idx=idx, q=q).mean("member")


## compute asymmetry
for v in ["sst", "pr"]:
    comps[f"{v}_asym"] = comps[f"{v}_warm"] + comps[f"{v}_cold"]
    comps_ref[f"{v}_asym"] = comps_ref[f"{v}_warm"] + comps_ref[f"{v}_cold"]

Reconstruct CESM2

In [ ]:
## normalize by El Niño
get_scale = lambda x: np.abs(src.utils.get_nino3(x["sst_warm"]).sel(season="OND"))
scale_ref = get_scale(comps_ref)
scale_anom = get_scale(comps)
scale_ratio = scale_ref / scale_anom

## apply scaling
comps_norm = scale_ratio * comps

print(scale_ratio.values.item())

## Plot

### funcs

In [ ]:
import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_sst2(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_sst(ax, data, amp=2):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance",
        levels=src.utils.make_cb_range(amp, amp / 5),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_pr(ax, data, amp=20):
    """plot data on ax"""

    ax.contour(
        data.longitude,
        data.latitude,
        data,
        colors="k",
        linewidths=0.8,
        alpha=0.75,
        levels=src.utils.make_cb_range(amp, amp / 5),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return

In [ ]:
## specify lons/lats for E/W boxes
LONS_E = [240, 280]
LONS_W = [150, 190]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-20, 0, 20],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[150, -170, -120, -80],
            ylocs=[-20, 0, 20],
        )
        gl_.top_labels = False
    gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return

### SST plot

In [ ]:
## specify how to select season
# sel = lambda x: x.mean("season")
sel = lambda x: x.sel(season="OND")

## specify intervals
dxs = np.array(
    [
        [0.5, 0.5],
        [0.5, 0.5],
        [0.2, 0.2],
    ],
)

fig = plt.figure(figsize=(10, 7.5), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=40, lon_range=(60, 285))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=2, format_func=format_func)


cp00 = plot_sst2(axs[0, 0], sel(comps_ref["sst_warm"]), dx=dxs[0, 0])
cp00 = plot_sst2(axs[0, 1], sel(comps_norm["sst_warm"]), dx=dxs[0, 1])

cp10 = plot_sst2(axs[1, 0], sel(comps_ref["sst_cold"]), dx=dxs[1, 0])
cp11 = plot_sst2(axs[1, 1], sel(comps_norm["sst_cold"]), dx=dxs[1, 1])

cp10 = plot_sst2(axs[2, 0], sel(comps_ref["sst_asym"]), dx=dxs[2, 0])
cp11 = plot_sst2(axs[2, 1], sel(comps_norm["sst_asym"]), dx=dxs[2, 1])

## add formatting to axs
for ax in axs.flatten():
    plot_wbox(ax, c="gray", lw=0.8)
    plot_ebox(ax, c="gray", lw=0.8)
    src.utils.plot_nino34_box(ax, c="w", ls="--", lw=0.8, alpha=0.75)

## label
axs[0, 0].set_title("(a) El Niño (ERSST)", loc="left")
axs[0, 1].set_title(r"(b) El Niño (CESM2, scaled)", loc="left")
axs[1, 0].set_title("(c) La Niña (ERSST)", loc="left")
axs[1, 1].set_title("(d) La Niña (CESM2, scaled)", loc="left")
axs[2, 0].set_title("(c) Asym. (ERSST)", loc="left")
axs[2, 1].set_title("(d) Asym. (CESM2, scaled)", loc="left")

## add labels
add_gridlines(axs)

## aspect
for ax in axs.flatten():
    ax.set_aspect("auto")

bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax, dx in zip(axs.flatten(), dxs.flatten()):
    ax.text(s=f"Interval: {dx} K", transform=ax.transAxes, **legend_kwargs)
    src.utils.plot_box(ax=ax, lons=[65, 140], lats=[5, 35], ls="--", c="k", lw=1)


plt.show()

### Precip plot

In [ ]:
## specify how to select season
sel = lambda x: x.mean("season")
# sel = lambda x : x.sel(season="JAS")
# sel = lambda x : x.sel(season="OND")

## specify intervals
dxs = 0.5 * np.ones([3, 2])

fig = plt.figure(figsize=(10, 7.5), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=40, lon_range=(60, 285))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=2, format_func=format_func)


cp00 = plot_sst2(axs[0, 0], -sel(comps_ref["pr_warm"]), dx=dxs[0, 0])
cp00 = plot_sst2(axs[0, 1], -sel(comps_norm["pr_warm"]), dx=dxs[0, 1])

cp10 = plot_sst2(axs[1, 0], -sel(comps_ref["pr_cold"]), dx=dxs[1, 0])
cp11 = plot_sst2(axs[1, 1], -sel(comps_norm["pr_cold"]), dx=dxs[1, 1])

cp10 = plot_sst2(axs[2, 0], -sel(comps_ref["pr_asym"]), dx=dxs[2, 0])
cp11 = plot_sst2(axs[2, 1], -sel(comps_norm["pr_asym"]), dx=dxs[2, 1])

## add formatting to axs
for ax in axs.flatten():
    plot_wbox(ax, c="gray", lw=0.8)
    plot_ebox(ax, c="gray", lw=0.8)
    src.utils.plot_nino34_box(ax, c="w", ls="--", lw=0.8, alpha=0.75)

## label
axs[0, 0].set_title("(a) El Niño (ERSST)", loc="left")
axs[0, 1].set_title(r"(b) El Niño (CESM2, scaled)", loc="left")
axs[1, 0].set_title("(c) La Niña (ERSST)", loc="left")
axs[1, 1].set_title("(d) La Niña (CESM2, scaled)", loc="left")
axs[2, 0].set_title("(c) Asym. (ERSST)", loc="left")
axs[2, 1].set_title("(d) Asym. (CESM2, scaled)", loc="left")


## add labels
add_gridlines(axs)

## aspect
for ax in axs.flatten():
    ax.set_aspect("auto")

bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax, dx in zip(axs.flatten(), dxs.flatten()):
    ax.text(s=f"Interval: {dx} K", transform=ax.transAxes, **legend_kwargs)
    # for lats in [[-35, -7.5], [7.5, 35]]:
    src.utils.plot_box(ax=ax, lons=[90, 120], lats=[7.5, 35], ls="--", c="k", lw=1)
    src.utils.plot_box(ax=ax, lons=[90, 150], lats=[-7.5, 7.5], ls="--", c="k", lw=1)
    src.utils.plot_box(ax=ax, lons=[110, 170], lats=[-35, -7.5], ls="--", c="k", lw=1)


plt.show()

In [ ]:
get_pr_n = lambda x: x.sel(latitude=slice(7.5, 35), longitude=slice(90, 120)).mean(
    ["longitude", "latitude"]
)

get_pr_s = lambda x: x.sel(latitude=slice(-35, -7.5), longitude=slice(110, 170)).mean(
    ["longitude", "latitude"]
)

get_pr_eq = lambda x: x.sel(latitude=slice(-7.5, 7.5), longitude=slice(90, 150)).mean(
    ["longitude", "latitude"]
)


def get_indices(x):
    """compute indices"""

    idxs = xr.merge(
        [
            get_pr_n(x["pr"]).rename("pr_n"),
            get_pr_s(x["pr"]).rename("pr_s"),
            get_pr_eq(x["pr"]).rename("pr_eq"),
            src.utils.get_nino3(x["sst"]).rename("T_3"),
            src.utils.get_nino34(x["sst"]).rename("T_34"),
            src.utils.get_nino4(x["sst"]).rename("T_4"),
        ]
    )

    return idxs


idxs = get_indices(anom)
idxs_ref = get_indices(anom_ref)

In [ ]:
## SPEcify t variable
T_VAR = "T_34"

## specify season to plot
# sel = lambda x : x.mean("season")
sel = lambda x: x.sel(season="JFM")

fig, axs = plt.subplots(1, 2, figsize=(7, 3.5), layout="constrained")
for ax, idxs_ in zip(axs, [idxs, idxs_ref]):

    ## plot data
    ax.scatter(sel(idxs_[T_VAR]), sel(idxs_["pr_n"]))
src.utils.set_lims(axs)

for ax in axs:
    kwargs = dict(ls="--", c="gray", lw=1)
    ax.axvline(0, **kwargs)
    ax.axhline(0, **kwargs)

plt.show()

fig, axs = plt.subplots(1, 2, figsize=(7, 3.5), layout="constrained")
for ax, idxs_ in zip(axs, [idxs, idxs_ref]):

    ## plot data
    ax.scatter(sel(idxs_[T_VAR]), sel(idxs_["pr_eq"]))
src.utils.set_lims(axs)

for ax in axs:
    kwargs = dict(ls="--", c="gray", lw=1)
    ax.axvline(0, **kwargs)
    ax.axhline(0, **kwargs)

plt.show()


fig, axs = plt.subplots(1, 2, figsize=(7, 3.5), layout="constrained")
for ax, idxs_ in zip(axs, [idxs, idxs_ref]):

    ## plot data
    ax.scatter(sel(idxs_[T_VAR]), sel(idxs_["pr_s"]))
src.utils.set_lims(axs)

for ax in axs:
    kwargs = dict(ls="--", c="gray", lw=1)
    ax.axvline(0, **kwargs)
    ax.axhline(0, **kwargs)

plt.show()